In [2]:
from sqlalchemy import create_engine, text, URL
from getpass import getpass

# Prompts you to type your postgres password — it is NOT stored in the notebook.
password = getpass("Postgres password: ")

# Note the port: 5433, your non-standard one.
url = URL.create(
    "postgresql+psycopg2",
    username="postgres",
    password=password,
    host="localhost",
    port=5433,
    database="detroit_property",
)
engine = create_engine(url)

with engine.connect() as conn:
    print("Connected!")
    print(conn.execute(text("SELECT version();")).scalar())
    print(conn.execute(text("SELECT postgis_full_version();")).scalar())

Connected!
PostgreSQL 16.2, compiled by Visual C++ build 1937, 64-bit
POSTGIS="3.4.1 3.4.1" [EXTENSION] PGSQL="160" GEOS="3.12.1-CAPI-1.18.1" PROJ="8.2.1 NETWORK_ENABLED=OFF URL_ENDPOINT=https://cdn.proj.org USER_WRITABLE_DIRECTORY=C:\WINDOWS\ServiceProfiles\NetworkService\AppData\Local/proj DATABASE_PATH=C:\Program Files\PostgreSQL\16\share\contrib\postgis-3.4\proj\proj.db" LIBXML="2.9.14" LIBJSON="0.12" LIBPROTOBUF="1.2.1" WAGYU="0.5.0 (Internal)"


In [3]:
import requests
import pandas as pd

# Paste your GeoService URL here (the part ending in /FeatureServer/0).
# If yours ends in just /FeatureServer, add /0 for the first layer.
LAYER_URL = "https://services2.arcgis.com/qvkbeam7Wirps6zC/arcgis/rest/services/assessor_property_sales_view/FeatureServer/0"
query_url = f"{LAYER_URL}/query"

params = {
    "where": "1=1",          # ArcGIS requires a filter; 1=1 means "everything"
    "outFields": "*",         # all columns
    "returnGeometry": "false",# skip geometry for now — this dataset is tabular
    "f": "json",              # ask for ArcGIS JSON back
    "resultRecordCount": 5,   # just 5 rows for the smoke test
}

r = requests.get(query_url, params=params, timeout=60)
r.raise_for_status()
data = r.json()

features = data["features"]
print(f"Got {len(features)} rows")

# Each feature is {"attributes": {...}} — pull the attributes into a DataFrame
df = pd.DataFrame([f["attributes"] for f in features])
df

Got 5 rows


,sale_id,parcel_id,address,sale_date,amt_sale_price,grantor,grantee,liber_page,term_of_sale,sale_verification,...,council_district,zip_code,street_number,street_prefix,street_name,street_type,unit_number,longitude,latitude,ObjectId
0,1355623,27070821.,8050 PIEDMONT,2011-06-27,3600,HUD,"DARBY, BRENT",NaN,19-MULTI PARCEL ARM'S LENGTH,PROPERTY TRANSFER AFFIDAVIT,...,None,None,None,None,None,None,None,None,None,21
1,1355625,27070821.,8050 PIEDMONT,2011-03-18,1,WAYNE COUNTY SHERIFF,HUD,NaN,19-MULTI PARCEL ARM'S LENGTH,PROPERTY TRANSFER AFFIDAVIT,...,None,None,None,None,None,None,None,None,None,23
2,1357110,27070408.,18266 HUBBELL,2011-04-01,11000,HUD,"LYNCH, YVETTE",49135:156-157,19-MULTI PARCEL ARM'S LENGTH,PROPERTY TRANSFER AFFIDAVIT,...,None,None,None,None,None,None,None,None,None,61
3,1311556,27090161.,20307 BLACKSTONE,2011-10-14,137619,WAYNE COUNTY SHERIFF,JPMORGAN CHASE BANK,NaN,19-MULTI PARCEL ARM'S LENGTH,PROPERTY TRANSFER AFFIDAVIT,...,None,None,None,None,None,None,None,None,None,106
4,1351212,27070509.,22615 TIREMAN,2011-03-08,10500,"BOGART, MARK RICHARD & LUCJA","WARD, CYNTHIA",NaN,19-MULTI PARCEL ARM'S LENGTH,PROPERTY TRANSFER AFFIDAVIT,...,None,None,None,None,None,None,None,None,None,218


In [4]:
import time

query_url = f"{LAYER_URL}/query"

# 1) Ask how many rows exist, so we know how many pages to fetch
count_params = {"where": "1=1", "returnCountOnly": "true", "f": "json"}
total = requests.get(query_url, params=count_params, timeout=60).json()["count"]
print(f"Total records to pull: {total:,}")

# 2) Page through the service. ArcGIS caps rows per request, so we loop with an offset.
PAGE = 1000
all_rows = []
offset = 0

while offset < total:
    params = {
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": "false",
        "f": "json",
        "resultOffset": offset,
        "resultRecordCount": PAGE,
        "orderByFields": "sale_id",   # stable sort so paging doesn't skip/dupe rows
    }
    resp = requests.get(query_url, params=params, timeout=120)
    resp.raise_for_status()
    feats = resp.json().get("features", [])
    if not feats:
        break
    all_rows.extend(f["attributes"] for f in feats)
    offset += PAGE
    print(f"  pulled {len(all_rows):,} / {total:,}", end="\r")
    time.sleep(0.2)   # be polite to a public API — don't hammer it

print(f"\nDone. Pulled {len(all_rows):,} rows.")
df = pd.DataFrame(all_rows)
print(df.shape)

Total records to pull: 514,384


ReadTimeout: HTTPSConnectionPool(host='services2.arcgis.com', port=443): Read timed out. (read timeout=120)

In [6]:
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# A session that automatically retries failed/slow requests with backoff
session = requests.Session()
retries = Retry(total=5, backoff_factor=1.0,
                status_forcelist=[429, 500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retries))

query_url = f"{LAYER_URL}/query"
total = session.get(query_url,
                    params={"where": "1=1", "returnCountOnly": "true", "f": "json"},
                    timeout=60).json()["count"]

# Resume: if all_rows already exists from the earlier run, keep it; else start fresh
try:
    all_rows
except NameError:
    all_rows = []

PAGE = 1000
offset = (len(all_rows) // PAGE) * PAGE   # resume on a clean page boundary
all_rows = all_rows[:offset]              # trim any partial page so we don't dupe

while offset < total:
    params = {
        "where": "1=1", "outFields": "*", "returnGeometry": "false", "f": "json",
        "resultOffset": offset, "resultRecordCount": PAGE, "orderByFields": "sale_id",
    }
    resp = session.get(query_url, params=params, timeout=180)  # longer timeout too
    resp.raise_for_status()
    feats = resp.json().get("features", [])
    if not feats:
        break
    all_rows.extend(f["attributes"] for f in feats)
    offset += PAGE
    print(f"  pulled {len(all_rows):,} / {total:,}", end="\r")

print(f"\nDone. Pulled {len(all_rows):,} rows.")
df = pd.DataFrame(all_rows)
print(df.shape)

  pulled 514,384 / 514,384
Done. Pulled 514,384 rows.
(514384, 28)


In [7]:
from sqlalchemy import text

# Create the landing schema if it's not there yet
with engine.begin() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS raw;"))

# Land the data as-is into raw.property_sales
df.to_sql(
    "property_sales",
    engine,
    schema="raw",
    if_exists="replace",   # replace = re-running the notebook reloads cleanly (idempotent)
    index=False,
    chunksize=1000,        # keep chunks small — see note below
    method="multi",
)

# Confirm it actually landed
with engine.connect() as conn:
    n = conn.execute(text("SELECT COUNT(*) FROM raw.property_sales;")).scalar()
print(f"raw.property_sales row count: {n:,}")

raw.property_sales row count: 514,384
